# LeetCode 1476 — Subrectangle Queries bằng Python

Link đề: https://leetcode.com/problems/subrectangle-queries/ · Độ khó: **Medium** · Chủ đề: Array, Design, Matrix

Skeleton LeetCode:

```python
class SubrectangleQueries:

    def __init__(self, rectangle: list[list[int]]):
        ...

    def updateSubrectangle(self, row1: int, col1: int, row2: int, col2: int, newValue: int) -> None:
        ...

    def getValue(self, row: int, col: int) -> int:
        ...


# Your SubrectangleQueries object will be instantiated and called as such:
# obj = SubrectangleQueries(rectangle)
# obj.updateSubrectangle(row1,col1,row2,col2,newValue)
# param_2 = obj.getValue(row,col)
```

> Bài **thiết kế lớp**: notebook tự dựng **bộ chạy chuỗi lệnh giả lập** kiểu LeetCode (danh sách tên lệnh + danh sách tham số → danh sách kết quả) để kiểm thử; đây là bản giả lập, không phải trình chấm thật.

## 1. Mục tiêu học

1. Thiết kế lớp giữ trạng thái một **ma trận** và hỗ trợ hai thao tác: **gán cả hình chữ nhật con** và **đọc một ô**.
2. Hiểu đánh đổi **cập nhật nhanh – truy vấn chậm** và ngược lại: cập nhật trực tiếp `O(rc)` / đọc `O(1)` so với **lưu lịch sử** cập nhật `O(1)` / đọc `O(U)` (duyệt ngược).
3. Biết **sao chép sâu** ma trận đúng cách (tránh dùng chung tham chiếu với dữ liệu đầu vào).
4. Kết hợp hai cách bằng **gộp định kỳ (flush)** — ý tưởng chia căn (sqrt) để cân bằng chi phí.
5. Viết **bộ chạy lệnh giả lập** và test đối chiếu ngẫu nhiên giữa nhiều cài đặt.

## 2. Đề bài & ví dụ

Cài đặt lớp `SubrectangleQueries` nhận một ma trận số nguyên `rows × cols` ở constructor và hỗ trợ:
1. `updateSubrectangle(row1, col1, row2, col2, newValue)`: gán **mọi ô** trong hình chữ nhật con có góc trên-trái `(row1, col1)` và góc dưới-phải `(row2, col2)` (bao gồm cả biên) bằng `newValue`.
2. `getValue(row, col)`: trả về giá trị hiện tại của ô `(row, col)`.

**Ví dụ 1:**

```
["SubrectangleQueries","getValue","updateSubrectangle","getValue","getValue","updateSubrectangle","getValue","getValue"]
[[[[1,2,1],[4,3,4],[3,2,1],[1,1,1]]],[0,2],[0,0,3,2,5],[0,2],[3,1],[3,0,3,2,10],[3,1],[0,2]]
→ [null,1,null,5,5,null,10,5]
```

Ma trận ban đầu 4×3; sau `update(0,0,3,2,5)` toàn bộ thành 5; sau `update(3,0,3,2,10)` hàng cuối thành 10.

**Ví dụ 2:**

```
["SubrectangleQueries","getValue","updateSubrectangle","getValue","getValue","updateSubrectangle","getValue"]
[[[[1,1,1],[2,2,2],[3,3,3]]],[0,0],[0,0,2,2,100],[0,0],[2,2],[1,1,2,2,20],[2,2]]
→ [null,1,null,100,100,null,20]
```

**Ràng buộc:**
- Tối đa `500` thao tác (`updateSubrectangle` + `getValue`).
- `1 <= rows, cols <= 100`; `0 <= row1 <= row2 < rows`; `0 <= col1 <= col2 < cols`
- `1 <= newValue, rectangle[i][j] <= 10^9`; `0 <= row < rows`, `0 <= col < cols`

In [1]:
import copy
import random
import time
from typing import List

EX1 = (["SubrectangleQueries", "getValue", "updateSubrectangle", "getValue", "getValue",
        "updateSubrectangle", "getValue", "getValue"],
       [[[[1, 2, 1], [4, 3, 4], [3, 2, 1], [1, 1, 1]]], [0, 2], [0, 0, 3, 2, 5], [0, 2], [3, 1],
        [3, 0, 3, 2, 10], [3, 1], [0, 2]],
       [None, 1, None, 5, 5, None, 10, 5])
EX2 = (["SubrectangleQueries", "getValue", "updateSubrectangle", "getValue", "getValue",
        "updateSubrectangle", "getValue"],
       [[[[1, 1, 1], [2, 2, 2], [3, 3, 3]]], [0, 0], [0, 0, 2, 2, 100], [0, 0], [2, 2],
        [1, 1, 2, 2, 20], [2, 2]],
       [None, 1, None, 100, 100, None, 20])
EXAMPLES = [EX1, EX2]

def run_ops(cls, ops, args):
    """Bộ chạy lệnh GIẢ LẬP kiểu LeetCode. Truyền bản sao sâu của tham số để các lần chạy độc lập."""
    args = copy.deepcopy(args)
    obj = cls(*args[0])
    out = [None]
    for name, a in zip(ops[1:], args[1:]):
        out.append(getattr(obj, name)(*a))
    return out

for ops, args, want in EXAMPLES:
    print(len(ops), "lệnh -> kỳ vọng", want)

8 lệnh -> kỳ vọng [None, 1, None, 5, 5, None, 10, 5]
7 lệnh -> kỳ vọng [None, 1, None, 100, 100, None, 20]


## 3. Kiến thức nền

### 3.1 Ma trận là list các list; truy cập `m[r][c]`

`m[r]` là **hàng** `r` (một list), `m[r][c]` là ô ở hàng `r`, cột `c`. `len(m)` là số hàng, `len(m[0])` là số cột.

In [2]:
m = [[1, 2, 1], [4, 3, 4], [3, 2, 1], [1, 1, 1]]
print("số hàng:", len(m), "| số cột:", len(m[0]), "| m[1][2] =", m[1][2])
for row in m:
    print(row)

số hàng: 4 | số cột: 3 | m[1][2] = 4
[1, 2, 1]
[4, 3, 4]
[3, 2, 1]
[1, 1, 1]


### 3.2 Sao chép nông vs sao chép sâu

`m2 = m` chỉ là **tên khác** của cùng đối tượng. `m[:]` hay `list(m)` chỉ sao chép **list ngoài** — các hàng vẫn dùng chung. Muốn độc lập phải sao từng hàng: `[row[:] for row in m]` (hoặc `copy.deepcopy`).

In [3]:
orig = [[1, 2], [3, 4]]
alias = orig
shallow = orig[:]
deep = [row[:] for row in orig]
orig[0][0] = 99
print("alias  :", alias)
print("shallow:", shallow, "<- hàng vẫn dùng chung!")
print("deep   :", deep)

alias  : [[99, 2], [3, 4]]
shallow: [[99, 2], [3, 4]] <- hàng vẫn dùng chung!
deep   : [[1, 2], [3, 4]]


### 3.3 Gán lát cắt (slice assignment)

`row[c1:c2+1] = [v] * (c2 - c1 + 1)` gán cả đoạn cột trong một lệnh (chạy trong C, nhanh hơn vòng `for` thuần Python).

In [4]:
row = [1, 2, 3, 4, 5]
row[1:4] = [9] * 3
print(row)

[1, 9, 9, 9, 5]


### 3.4 Duyệt ngược danh sách: `reversed()`

`for x in reversed(lst)` đi từ phần tử **mới nhất** về cũ nhất mà không tạo list mới — dùng cho cách "lưu lịch sử": bản cập nhật **gần nhất** phủ ô sẽ quyết định giá trị.

In [5]:
hist = ["cũ", "giữa", "mới"]
for x in reversed(hist):
    print(x, end=" ")
print()

mới giữa cũ 


### 3.5 Kiểm tra một điểm nằm trong hình chữ nhật

Ô `(r, c)` thuộc hình `(r1, c1, r2, c2)` ⇔ `r1 <= r <= r2 and c1 <= c <= c2`. Python cho phép viết so sánh **chuỗi** `a <= x <= b`.

In [6]:
def inside(r, c, r1, c1, r2, c2):
    return r1 <= r <= r2 and c1 <= c <= c2

print(inside(3, 1, 3, 0, 3, 2), inside(0, 2, 3, 0, 3, 2), inside(2, 2, 1, 1, 2, 2))

True False True


### 3.6 `getattr` để gọi method theo tên

Bộ chạy lệnh nhận tên method là chuỗi (`"getValue"`), `getattr(obj, name)` lấy ra method để gọi.

In [7]:
class Demo:
    def hello(self, x):
        return f"hello {x}"
print(getattr(Demo(), "hello")("world"))

hello world


## 4. Ý tưởng nền tảng / trực giác

Mô phỏng tay ví dụ 1:

```
ban đầu        update(0,0,3,2,5)    update(3,0,3,2,10)
1 2 1          5 5 5                5  5  5
4 3 4    ->    5 5 5         ->     5  5  5
3 2 1          5 5 5                5  5  5
1 1 1          5 5 5                10 10 10
```

- `getValue(0,2)` trước update: `1`; sau update 1: `5`; `getValue(3,1)`: `5`; sau update 2: `getValue(3,1) = 10`, `getValue(0,2) = 5`.

Hai cách nhìn:
1. **Sơn trực tiếp** lên ma trận mỗi lần update (tốn công khi update, đọc tức thì).
2. **Chỉ ghi lại lệnh sơn**; khi cần đọc ô `(r, c)`, hỏi: "lệnh sơn **gần nhất** nào phủ ô này?" — nếu không có thì là giá trị gốc.

In [8]:
grid = [row[:] for row in EX1[1][0][0]]
for (r1, c1, r2, c2, v) in [(0, 0, 3, 2, 5), (3, 0, 3, 2, 10)]:
    for r in range(r1, r2 + 1):
        for c in range(c1, c2 + 1):
            grid[r][c] = v
    print(f"sau update {(r1, c1, r2, c2, v)}:")
    for row in grid:
        print("   ", row)

sau update (0, 0, 3, 2, 5):
    [5, 5, 5]
    [5, 5, 5]
    [5, 5, 5]
    [5, 5, 5]
sau update (3, 0, 3, 2, 10):
    [5, 5, 5]
    [5, 5, 5]
    [5, 5, 5]
    [10, 10, 10]


## 5. Cách 1 — Brute Force / baseline: cập nhật trực tiếp bằng hai vòng lặp

**Ý tưởng:** sao chép sâu ma trận vào `self.rect`. `updateSubrectangle` gán từng ô bằng hai vòng `for`; `getValue` đọc `self.rect[row][col]`. Đây là đúng gợi ý của đề.

In [9]:
class SubrectBrute:
    def __init__(self, rectangle: List[List[int]]):
        self.rect = [row[:] for row in rectangle]      # sao chép sâu

    def updateSubrectangle(self, row1, col1, row2, col2, newValue) -> None:
        for r in range(row1, row2 + 1):
            for c in range(col1, col2 + 1):
                self.rect[r][c] = newValue

    def getValue(self, row, col) -> int:
        return self.rect[row][col]

for ops, args, want in EXAMPLES:
    got = run_ops(SubrectBrute, ops, args)
    print(got, got == want)

[None, 1, None, 5, 5, None, 10, 5] True
[None, 1, None, 100, 100, None, 20] True


**Độ phức tạp:** `update` `O((row2−row1+1)·(col2−col1+1))` ≤ `O(rows·cols) = 10^4`; `getValue` `O(1)`; bộ nhớ `O(rows·cols)`. Với ≤ 500 thao tác: tối đa `5·10^6` phép gán — vẫn ổn.

**Nút thắt:** nếu ma trận lớn hơn nhiều (ví dụ `10^3 × 10^3`) và có nhiều update phủ rộng, mỗi update quá tốn kém trong khi có thể **không bao giờ** có ai đọc phần lớn các ô vừa sơn. Đo thử:

In [10]:
def random_ops(rows, cols, n_ops, rng, p_update=0.5, max_val=10**9):
    rect = [[rng.randint(1, max_val) for _ in range(cols)] for _ in range(rows)]
    ops, args = ["SubrectangleQueries"], [[rect]]
    for _ in range(n_ops):
        if rng.random() < p_update:
            r1, r2 = sorted(rng.randrange(rows) for _ in range(2))
            c1, c2 = sorted(rng.randrange(cols) for _ in range(2))
            ops.append("updateSubrectangle"); args.append([r1, c1, r2, c2, rng.randint(1, max_val)])
        else:
            ops.append("getValue"); args.append([rng.randrange(rows), rng.randrange(cols)])
    return ops, args

rng = random.Random(1476)
for size in (100, 200, 400):
    ops, args = random_ops(size, size, 500, rng)
    t = time.perf_counter()
    run_ops(SubrectBrute, ops, args)
    print(f"{size}x{size}, 500 thao tác: {time.perf_counter() - t:.3f}s")

100x100, 500 thao tác: 0.010s
200x200, 500 thao tác: 0.030s


400x400, 500 thao tác: 0.145s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** giá trị của ô `(r, c)` tại một thời điểm = `newValue` của **lệnh update gần nhất có phủ `(r, c)`**; nếu không lệnh nào phủ thì là giá trị gốc. Vậy không cần sơn ngay — chỉ cần **ghi lại** danh sách lệnh.

**Quy tắc:**
- `update(...)`: **nếu** dùng lịch sử **thì** chỉ `append((r1, c1, r2, c2, v))` — `O(1)`.
- `getValue(r, c)`: duyệt lịch sử **từ mới về cũ**; **nếu** gặp lệnh phủ `(r, c)` **thì** trả `v` ngay; hết lịch sử **thì** trả `rect[r][c]` gốc.

**Đánh đổi:** cập nhật `O(1)`, đọc `O(U)` (U = số update). Có thể **cân bằng**: khi lịch sử dài quá ngưỡng `B`, "đổ" (flush) lịch sử vào ma trận một lần rồi xóa — đọc tối đa `O(B)`.

**Invariant (lịch sử):** `rect` gốc + danh sách lệnh theo thứ tự thời gian luôn mô tả đúng ma trận hiện tại.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Cập nhật trực tiếp bằng gán lát cắt theo hàng

Vẫn là cập nhật trực tiếp nhưng thay vòng `for` cột bằng **gán lát cắt** — cùng độ phức tạp `O(rc)` nhưng hằng số nhỏ hơn nhiều trong Python.

**Từng đoạn:**
1. Tạo sẵn `vals = [newValue] * (col2 − col1 + 1)`.
2. Với mỗi hàng `r` trong `[row1, row2]`: `self.rect[r][col1:col2 + 1] = vals` (Python sao chép phần tử, các hàng không dùng chung list `vals`).

In [11]:
class SubrectSlice:
    def __init__(self, rectangle: List[List[int]]):
        self.rect = [row[:] for row in rectangle]

    def updateSubrectangle(self, row1, col1, row2, col2, newValue) -> None:
        vals = [newValue] * (col2 - col1 + 1)
        for r in range(row1, row2 + 1):
            self.rect[r][col1:col2 + 1] = vals

    def getValue(self, row, col) -> int:
        return self.rect[row][col]

for ops, args, want in EXAMPLES:
    print(run_ops(SubrectSlice, ops, args) == want)

True
True


Code nộp LeetCode (Cách 2):

In [12]:
class SubrectangleQueries:

    def __init__(self, rectangle: List[List[int]]):
        self.rect = [row[:] for row in rectangle]

    def updateSubrectangle(self, row1: int, col1: int, row2: int, col2: int, newValue: int) -> None:
        vals = [newValue] * (col2 - col1 + 1)
        for r in range(row1, row2 + 1):
            self.rect[r][col1:col2 + 1] = vals

    def getValue(self, row: int, col: int) -> int:
        return self.rect[row][col]

for ops, args, want in EXAMPLES:
    print(run_ops(SubrectangleQueries, ops, args))

[None, 1, None, 5, 5, None, 10, 5]
[None, 1, None, 100, 100, None, 20]


**Độ phức tạp:** `update` `O(rows · cols)` (nhưng phần cột chạy trong C), `getValue` `O(1)`, bộ nhớ `O(rows · cols)`.

### 7.2 Cách 3 — Lưu lịch sử cập nhật, duyệt ngược khi đọc

**Từng đoạn:**
1. Constructor: lưu `self.rect` (sao chép sâu để không phụ thuộc đầu vào) và `self.hist = []`.
2. `updateSubrectangle`: `self.hist.append((row1, col1, row2, col2, newValue))`.
3. `getValue`: `for r1, c1, r2, c2, v in reversed(self.hist)`: nếu phủ thì `return v`; cuối cùng `return self.rect[row][col]`.

In [13]:
class SubrectHistory:
    def __init__(self, rectangle: List[List[int]]):
        self.rect = [row[:] for row in rectangle]
        self.hist = []

    def updateSubrectangle(self, row1, col1, row2, col2, newValue) -> None:
        self.hist.append((row1, col1, row2, col2, newValue))

    def getValue(self, row, col) -> int:
        for r1, c1, r2, c2, v in reversed(self.hist):
            if r1 <= row <= r2 and c1 <= col <= c2:
                return v
        return self.rect[row][col]

for ops, args, want in EXAMPLES:
    print(run_ops(SubrectHistory, ops, args) == want)

True
True


Code nộp LeetCode (Cách 3):

In [14]:
class SubrectangleQueries:

    def __init__(self, rectangle: List[List[int]]):
        self.rect = [row[:] for row in rectangle]
        self.hist = []

    def updateSubrectangle(self, row1: int, col1: int, row2: int, col2: int, newValue: int) -> None:
        self.hist.append((row1, col1, row2, col2, newValue))

    def getValue(self, row: int, col: int) -> int:
        for r1, c1, r2, c2, v in reversed(self.hist):
            if r1 <= row <= r2 and c1 <= col <= c2:
                return v
        return self.rect[row][col]

for ops, args, want in EXAMPLES:
    print(run_ops(SubrectangleQueries, ops, args))

[None, 1, None, 5, 5, None, 10, 5]
[None, 1, None, 100, 100, None, 20]


**Độ phức tạp:** `update` `O(1)`, `getValue` `O(U)` (U = số update đã có, ≤ 500), bộ nhớ `O(rows·cols + U)`. Tốt khi update phủ rộng mà ít truy vấn.

### 7.3 Cách 4 — Lịch sử + gộp định kỳ (flush) khi vượt ngưỡng

Kết hợp hai cách: lịch sử tối đa `B` lệnh; khi đầy, áp tất cả vào ma trận (theo thứ tự cũ → mới) rồi xóa lịch sử.
- `update`: `O(1)` + thỉnh thoảng flush `O(B · rows · cols)` ⇒ phân bổ `O(rows · cols)` mỗi update trong trường hợp xấu, nhưng thực tế nhanh hơn vì các lệnh có thể chồng nhau.
- `getValue`: `O(B)`.

Ý nghĩa chính là tư duy **chia căn / gom lô** — dùng nhiều trong bài khó hơn (khi cả update lẫn query đều nhiều).

In [15]:
class SubrectFlush:
    def __init__(self, rectangle: List[List[int]], B: int = 32):
        self.rect = [row[:] for row in rectangle]
        self.hist = []
        self.B = B

    def _flush(self):
        for r1, c1, r2, c2, v in self.hist:              # cũ -> mới
            vals = [v] * (c2 - c1 + 1)
            for r in range(r1, r2 + 1):
                self.rect[r][c1:c2 + 1] = vals
        self.hist.clear()

    def updateSubrectangle(self, row1, col1, row2, col2, newValue) -> None:
        self.hist.append((row1, col1, row2, col2, newValue))
        if len(self.hist) >= self.B:
            self._flush()

    def getValue(self, row, col) -> int:
        for r1, c1, r2, c2, v in reversed(self.hist):
            if r1 <= row <= r2 and c1 <= col <= c2:
                return v
        return self.rect[row][col]

for ops, args, want in EXAMPLES:
    print(run_ops(SubrectFlush, ops, args) == want, run_ops(lambda m: SubrectFlush(m, B=1), ops, args) == want)

True True
True True


Code nộp LeetCode (Cách 4):

In [16]:
class SubrectangleQueries:

    def __init__(self, rectangle: List[List[int]]):
        self.rect = [row[:] for row in rectangle]
        self.hist = []
        self.B = 32

    def updateSubrectangle(self, row1: int, col1: int, row2: int, col2: int, newValue: int) -> None:
        self.hist.append((row1, col1, row2, col2, newValue))
        if len(self.hist) >= self.B:
            for r1, c1, r2, c2, v in self.hist:
                vals = [v] * (c2 - c1 + 1)
                for r in range(r1, r2 + 1):
                    self.rect[r][c1:c2 + 1] = vals
            self.hist.clear()

    def getValue(self, row: int, col: int) -> int:
        for r1, c1, r2, c2, v in reversed(self.hist):
            if r1 <= row <= r2 and c1 <= col <= c2:
                return v
        return self.rect[row][col]

for ops, args, want in EXAMPLES:
    print(run_ops(SubrectangleQueries, ops, args))

[None, 1, None, 5, 5, None, 10, 5]
[None, 1, None, 100, 100, None, 20]


So sánh thời gian trên ma trận lớn hơn ràng buộc (để thấy khác biệt) với tỉ lệ update / query khác nhau:

In [17]:
IMPLS = [SubrectBrute, SubrectSlice, SubrectHistory, SubrectFlush]
for p_up in (0.9, 0.5, 0.1):
    rng = random.Random(int(p_up * 100))
    ops, args = random_ops(200, 200, 1500, rng, p_update=p_up)
    line = []
    for cls in IMPLS:
        t = time.perf_counter()
        run_ops(cls, ops, args)
        line.append(f"{cls.__name__[7:]}={time.perf_counter() - t:.3f}s")
    print(f"200x200, 1500 thao tác, {int(p_up * 100)}% update:", ", ".join(line))

200x200, 1500 thao tác, 90% update: Brute=0.153s, Slice=0.043s, History=0.011s, Flush=0.039s
200x200, 1500 thao tác, 50% update: Brute=0.068s, Slice=0.025s, History=0.011s, Flush=0.025s


200x200, 1500 thao tác, 10% update: Brute=0.021s, Slice=0.012s, History=0.010s, Flush=0.014s


## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1. Lịch sử sau từng lệnh và cách `getValue` duyệt ngược:

| lệnh | hist | getValue duyệt | trả về |
|---|---|---|---|
| get(0,2) | [] | không có lệnh → gốc `rect[0][2]` | 1 |
| update(0,0,3,2,5) | [U1] | | |
| get(0,2) | [U1] | U1 phủ ✔ | 5 |
| get(3,1) | [U1] | U1 phủ ✔ | 5 |
| update(3,0,3,2,10) | [U1, U2] | | |
| get(3,1) | [U1, U2] | U2 phủ ✔ (dừng ngay) | 10 |
| get(0,2) | [U1, U2] | U2 ✘ (hàng 0 ∉ [3,3]) → U1 ✔ | 5 |

In [18]:
class TracedHistory(SubrectHistory):
    def updateSubrectangle(self, row1, col1, row2, col2, newValue):
        super().updateSubrectangle(row1, col1, row2, col2, newValue)
        print(f"update{(row1, col1, row2, col2, newValue)} -> hist có {len(self.hist)} lệnh")

    def getValue(self, row, col):
        print(f"getValue({row},{col}):")
        for idx in range(len(self.hist) - 1, -1, -1):
            r1, c1, r2, c2, v = self.hist[idx]
            hit = r1 <= row <= r2 and c1 <= col <= c2
            print(f"   xét U{idx + 1}={self.hist[idx]}: {'PHỦ -> trả ' + str(v) if hit else 'không phủ'}")
            if hit:
                return v
        print(f"   hết lịch sử -> giá trị gốc {self.rect[row][col]}")
        return self.rect[row][col]

out = run_ops(TracedHistory, *EX1[:2])
print("kết quả:", out, "| khớp:", out == EX1[2])

getValue(0,2):
   hết lịch sử -> giá trị gốc 1
update(0, 0, 3, 2, 5) -> hist có 1 lệnh
getValue(0,2):
   xét U1=(0, 0, 3, 2, 5): PHỦ -> trả 5
getValue(3,1):
   xét U1=(0, 0, 3, 2, 5): PHỦ -> trả 5
update(3, 0, 3, 2, 10) -> hist có 2 lệnh
getValue(3,1):
   xét U2=(3, 0, 3, 2, 10): PHỦ -> trả 10
getValue(0,2):
   xét U2=(3, 0, 3, 2, 10): không phủ
   xét U1=(0, 0, 3, 2, 5): PHỦ -> trả 5
kết quả: [None, 1, None, 5, 5, None, 10, 5] | khớp: True


Trace Cách 1 (in ma trận sau mỗi update) trên ví dụ 2:

In [19]:
class TracedBrute(SubrectBrute):
    def updateSubrectangle(self, row1, col1, row2, col2, newValue):
        super().updateSubrectangle(row1, col1, row2, col2, newValue)
        print(f"update{(row1, col1, row2, col2, newValue)}:", self.rect)
    def getValue(self, row, col):
        v = super().getValue(row, col)
        print(f"getValue({row},{col}) = {v}")
        return v

_ = run_ops(TracedBrute, *EX2[:2])

getValue(0,0) = 1
update(0, 0, 2, 2, 100): [[100, 100, 100], [100, 100, 100], [100, 100, 100]]
getValue(0,0) = 100
getValue(2,2) = 100
update(1, 1, 2, 2, 20): [[100, 100, 100], [100, 20, 20], [100, 20, 20]]
getValue(2,2) = 20


## 9. Code nộp LeetCode

Bản khuyên dùng: **Cách 3 (lưu lịch sử)** — ngắn, `update` `O(1)`, `getValue` `O(U) ≤ 500`, đúng tên lớp và chữ ký. (Cách 2 cũng là lựa chọn tốt nếu truy vấn chiếm đa số.)

In [20]:
class SubrectangleQueries:

    def __init__(self, rectangle: List[List[int]]):
        self.rect = [row[:] for row in rectangle]   # không dùng chung với đầu vào
        self.hist = []                              # các lệnh update theo thời gian

    def updateSubrectangle(self, row1: int, col1: int, row2: int, col2: int, newValue: int) -> None:
        self.hist.append((row1, col1, row2, col2, newValue))

    def getValue(self, row: int, col: int) -> int:
        for r1, c1, r2, c2, v in reversed(self.hist):   # mới nhất trước
            if r1 <= row <= r2 and c1 <= col <= c2:
                return v
        return self.rect[row][col]

for ops, args, want in EXAMPLES:
    assert run_ops(SubrectangleQueries, ops, args) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

- **Cách 1, 2:** sau mỗi `update`, `self.rect` đúng bằng ma trận theo định nghĩa (mọi ô trong hình được gán, ô ngoài giữ nguyên). Gán lát cắt `rect[r][c1:c2+1] = vals` thay đúng các phần tử cột `c1..c2` của hàng `r`, độ dài không đổi.
- **Cách 3:** Gọi `U1, …, Ut` là các update theo thời gian. Giá trị thật của ô `(r, c)` là `v` của `Uj` **lớn nhất** (gần nhất) phủ `(r, c)` — vì mọi update sau đó không chạm ô này, còn `Uj` ghi đè mọi giá trị trước. Nếu không `Uj` nào phủ, ô giữ giá trị gốc. Duyệt ngược và dừng ở lệnh phủ đầu tiên đúng là tìm `j` lớn nhất.
- **Cách 4:** Bất biến: (ma trận `rect`) + (lịch sử hiện tại) = trạng thái thật. Flush áp các lệnh theo đúng thứ tự cũ → mới (như Cách 2) rồi xóa ⇒ bất biến giữ nguyên; `getValue` như Cách 3 trên phần lịch sử còn lại.

## 11. Phân tích Time / Space

Gọi `R × Cc` là kích thước ma trận (≤ 100 × 100), `U` số update, `Q` số truy vấn (`U + Q ≤ 500`).

| Cách | update | getValue | Tổng xấu nhất | Space |
|---|---|---|---|---|
| 1. Hai vòng lặp | `O(R·Cc)` | `O(1)` | `O(U·R·Cc + Q)` | `O(R·Cc)` |
| 2. Gán lát cắt | `O(R·Cc)` (hằng số nhỏ) | `O(1)` | như trên | `O(R·Cc)` |
| 3. Lịch sử | `O(1)` | `O(U)` | `O(U + Q·U)` ≤ `~6·10^4` | `O(R·Cc + U)` |
| 4. Lịch sử + flush | `O(1)` phân bổ + flush | `O(B)` | `O(U·R·Cc/…)` + `O(Q·B)` | `O(R·Cc + B)` |

Với ràng buộc của đề, Cách 3 có tổng chi phí xấu nhất nhỏ nhất (`250 × 250` phép so sánh).

## 12. Bộ test

Ví dụ đề qua bộ chạy lệnh giả lập cho mọi cài đặt; edge case (ma trận 1×1, update toàn ma trận, update chồng nhau, update rồi get ngoài vùng); đối chiếu ngẫu nhiên với Cách 1 (seed cố định, đúng ràng buộc); kiểm tra **đầu vào không bị thay đổi**.

In [21]:
ALL = IMPLS + [lambda m: SubrectFlush(m, B=1), lambda m: SubrectFlush(m, B=3), SubrectangleQueries]
for ops, args, want in EXAMPLES:
    for cls in ALL:
        assert run_ops(cls, ops, args) == want

edge = [
    (["SubrectangleQueries", "getValue", "updateSubrectangle", "getValue"],
     [[[[7]]], [0, 0], [0, 0, 0, 0, 3], [0, 0]], [None, 7, None, 3]),
    (["SubrectangleQueries", "updateSubrectangle", "updateSubrectangle", "getValue", "getValue", "getValue"],
     [[[[1, 1, 1], [1, 1, 1]]], [0, 0, 1, 2, 5], [0, 1, 1, 1, 9], [0, 0], [1, 1], [1, 2]],
     [None, None, None, 5, 9, 5]),
    (["SubrectangleQueries", "updateSubrectangle", "getValue", "getValue"],
     [[[[1, 2], [3, 4], [5, 6]]], [1, 0, 1, 1, 8], [0, 1], [2, 0]], [None, None, 2, 5]),
]
for ops, args, want in edge:
    for cls in ALL:
        assert run_ops(cls, ops, args) == want, (ops, args)
print("Ví dụ + edge case: OK")

Ví dụ + edge case: OK


In [22]:
rng = random.Random(2025)
for _ in range(300):
    rows, cols = rng.randint(1, 8), rng.randint(1, 8)
    ops, args = random_ops(rows, cols, rng.randint(1, 60), rng, p_update=rng.random(), max_val=rng.choice([3, 10**9]))
    want = run_ops(SubrectBrute, ops, args)
    for cls in ALL:
        assert run_ops(cls, ops, args) == want
# đầu vào không bị thay đổi
m0 = [[1, 2], [3, 4]]
for cls in ALL:
    q = cls(m0)
    q.updateSubrectangle(0, 0, 1, 1, 9)
    q.getValue(0, 0)
assert m0 == [[1, 2], [3, 4]]
print("Random 300 chuỗi lệnh: OK | đầu vào không bị thay đổi: OK")

ops, args = random_ops(100, 100, 500, random.Random(1))
t = time.perf_counter()
r = run_ops(SubrectangleQueries, ops, args)
print(f"Ràng buộc tối đa (100x100, 500 thao tác): {time.perf_counter() - t:.4f}s, khớp brute: {r == run_ops(SubrectBrute, ops, args)}")

Random 300 chuỗi lệnh: OK | đầu vào không bị thay đổi: OK
Ràng buộc tối đa (100x100, 500 thao tác): 0.0052s, khớp brute: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Gán thẳng `self.rect = rectangle` (dùng chung tham chiếu)

Khi update, ma trận **của người gọi** cũng bị sửa. Nếu hai đối tượng được tạo từ cùng một ma trận, update ở đối tượng này làm sai đối tượng kia.

In [23]:
class WrongAlias(SubrectBrute):
    def __init__(self, rectangle):
        self.rect = rectangle                          # SAI: không sao chép

shared = [[1, 2], [3, 4]]
q1, q2 = WrongAlias(shared), WrongAlias(shared)
q1.updateSubrectangle(0, 0, 1, 1, 100)
print("SAI : q2.getValue(0,0) =", q2.getValue(0, 0), "| đúng: 1 | ma trận gốc bị sửa:", shared)

SAI : q2.getValue(0,0) = 100 | đúng: 1 | ma trận gốc bị sửa: [[100, 100], [100, 100]]


**Sửa:** `self.rect = [row[:] for row in rectangle]`.

### Lỗi 2 — Sao chép nông `rectangle[:]`

Chỉ sao list ngoài; các hàng vẫn là cùng đối tượng.

In [24]:
class WrongShallow(SubrectBrute):
    def __init__(self, rectangle):
        self.rect = rectangle[:]                       # SAI: hàng vẫn dùng chung

shared = [[1, 2], [3, 4]]
q1, q2 = WrongShallow(shared), WrongShallow(shared)
q1.updateSubrectangle(1, 1, 1, 1, 50)
print("SAI : q2.getValue(1,1) =", q2.getValue(1, 1), "| đúng: 4")

SAI : q2.getValue(1,1) = 50 | đúng: 4


**Sửa:** sao từng hàng (`row[:]`) hoặc `copy.deepcopy`.

### Lỗi 3 — Quên biên: `range(row1, row2)` thay vì `range(row1, row2 + 1)`

Đề nói góc dưới-phải `(row2, col2)` **thuộc** hình chữ nhật.

In [25]:
class WrongExclusive(SubrectBrute):
    def updateSubrectangle(self, row1, col1, row2, col2, newValue):
        for r in range(row1, row2):                    # SAI: thiếu hàng row2
            for c in range(col1, col2):                # SAI: thiếu cột col2
                self.rect[r][c] = newValue

for ops, args, want in EXAMPLES:
    print("SAI :", run_ops(WrongExclusive, ops, args), "| đúng:", want)

SAI : [None, 1, None, 1, 1, None, 1, 1] | đúng: [None, 1, None, 5, 5, None, 10, 5]
SAI : [None, 1, None, 100, 3, None, 3] | đúng: [None, 1, None, 100, 100, None, 20]


**Sửa:** `range(row1, row2 + 1)` và `range(col1, col2 + 1)`.

### Lỗi 4 — Lịch sử duyệt **xuôi** (từ cũ tới mới)

Trả về lệnh **cũ nhất** phủ ô, trong khi lệnh mới hơn đã ghi đè.

In [26]:
class WrongForward(SubrectHistory):
    def getValue(self, row, col):
        for r1, c1, r2, c2, v in self.hist:            # SAI: phải reversed
            if r1 <= row <= r2 and c1 <= col <= c2:
                return v
        return self.rect[row][col]

ops, args, want = EX1
print("SAI :", run_ops(WrongForward, ops, args), "| đúng:", want)

SAI : [None, 1, None, 5, 5, None, 5, 5] | đúng: [None, 1, None, 5, 5, None, 10, 5]


`getValue(3,1)` sau update thứ hai phải là `10` nhưng bản sai gặp update thứ nhất trước nên trả `5`. **Sửa:** `reversed(self.hist)`.

### Lỗi 5 — Nhầm hàng/cột khi kiểm tra phủ

So `row` với khoảng cột hoặc ngược lại — chỉ lộ ra khi hình chữ nhật không vuông.

In [27]:
class WrongSwap(SubrectHistory):
    def getValue(self, row, col):
        for r1, c1, r2, c2, v in reversed(self.hist):
            if c1 <= row <= c2 and r1 <= col <= r2:    # SAI: đổi chỗ hàng/cột
                return v
        return self.rect[row][col]

ops, args, want = EX1
print("SAI :", run_ops(WrongSwap, ops, args), "| đúng:", want)

SAI : [None, 1, None, 5, 1, None, 1, 5] | đúng: [None, 1, None, 5, 5, None, 10, 5]


**Sửa:** `r1 <= row <= r2 and c1 <= col <= c2`. Test với hình chữ nhật **không vuông** để bắt lỗi này.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài thiết kế có **hai thao tác đối lập** (update vùng / query điểm) ⇒ hỏi ngay: thao tác nào nhiều hơn? Đẩy chi phí về phía thao tác **ít** hơn.
- "Gán cả vùng, đọc một điểm" với số thao tác nhỏ ⇒ **lưu lịch sử, duyệt ngược** (lệnh gần nhất thắng).
- Khi cả update lẫn query đều nhiều ⇒ nghĩ tới **gom lô / chia căn** hoặc cấu trúc nâng cao (segment tree 2D lazy, Fenwick 2D cho cộng dồn).
- Luôn **sao chép sâu** dữ liệu đầu vào khi lớp sẽ sửa nó.
- Test với ma trận **không vuông** và vùng **một ô** để bắt lỗi đổi chỉ số, lỗi biên.

## 15. Biến thể của bài

1. **Cộng thêm** `delta` vào hình chữ nhật thay vì gán: dùng **mảng hiệu 2D** (difference array) nếu truy vấn sau cùng, hoặc Fenwick 2D cho xen kẽ.
2. **Truy vấn tổng** hình chữ nhật (304 Range Sum Query 2D – Immutable / 308 Mutable): prefix sum 2D / Fenwick 2D.
3. **Undo update gần nhất** — bài tập 5 (lịch sử làm việc này rất tự nhiên).
4. **Đọc cả một hàng** hiện tại (bài tập 3).
5. **Ma trận rất lớn, rất ít ô khác gốc:** chỉ lưu lịch sử, không lưu ma trận (nếu ma trận gốc cho bằng công thức).

## 16. So sánh các cách

| Cách | Ý tưởng | update | getValue | Space | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Hai vòng lặp | Sơn từng ô | `O(R·Cc)` | `O(1)` | `O(R·Cc)` | Gợi ý đề, chuẩn đối chiếu |
| 2. Gán lát cắt | Sơn từng hàng bằng slice | `O(R·Cc)` (nhanh) | `O(1)` | `O(R·Cc)` | Nhiều truy vấn |
| 3. Lịch sử | Ghi lệnh, duyệt ngược | `O(1)` | `O(U)` | `O(R·Cc + U)` | Ít truy vấn / update rộng; khuyên dùng |
| 4. Lịch sử + flush | Gom lô `B` lệnh rồi sơn | `O(1)` + flush | `O(B)` | `O(R·Cc + B)` | Cân bằng khi cả hai đều nhiều |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** Ma trận `[[1,2],[3,4]]`; thực hiện `update(0,1,1,1,7)`, `getValue(1,1)`, `update(0,0,0,1,0)`, `getValue(0,1)`, `getValue(1,0)`. Ba giá trị trả về là gì?

**Bài 2 (tính tay):** Viết ma trận sau mỗi update của bài 1.

**Bài 3 (điền TODO):** Thêm vào Cách 3 method `getRow(row)` trả về **cả hàng** `row` hiện tại (list độ dài `cols`).

**Bài 4 (tự viết lại):** Không nhìn code, viết lại Cách 2 (gán lát cắt) thành lớp `MySliceQueries`, đối chiếu ngẫu nhiên với Cách 1.

**Bài 5 (mini challenge):** Viết `SubrectUndo` hỗ trợ thêm `undo()`: hủy lệnh `updateSubrectangle` **gần nhất chưa bị hủy** (không làm gì nếu không còn). Kiểm tra bằng bản tham chiếu lưu ảnh chụp ma trận.

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
class HistoryWithRowTodo(SubrectHistory):
    def getRow(self, row: int) -> List[int]:
        # TODO: với mỗi cột, tìm lệnh gần nhất phủ (row, col)
        raise NotImplementedError

# Bài 5 — TODO
class SubrectUndoTodo:
    def __init__(self, rectangle: List[List[int]]):
        raise NotImplementedError
    def updateSubrectangle(self, row1, col1, row2, col2, newValue) -> None:
        raise NotImplementedError
    def getValue(self, row, col) -> int:
        raise NotImplementedError
    def undo(self) -> None:
        # TODO: gợi ý — với cách lưu lịch sử, undo chỉ là pop
        raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Update đầu sơn cột 1 cả hai hàng. (2) Update hai sơn hàng 0. (3) `(1,0)` chưa bị sơn lần nào.

**Bài 2.** (1) `[[1,7],[3,7]]`. (2) Rồi `[[0,0],[3,7]]`. (3) So với kết quả bài 1.

**Bài 3.** (1) Duyệt `col` từ `0` tới `cols−1`. (2) Gọi lại `self.getValue(row, col)` là đúng nhưng `O(cols · U)`. (3) Tối ưu: bắt đầu từ bản sao hàng gốc, áp các lệnh phủ hàng `row` theo thứ tự cũ → mới.

**Bài 4.** (1) Sao chép sâu ở constructor. (2) `vals = [v] * (c2 − c1 + 1)`. (3) Đối chiếu bằng `random_ops` + `run_ops`.

**Bài 5.** (1) Dùng Cách 3: `undo` là `if self.hist: self.hist.pop()`. (2) Bản tham chiếu: mỗi update đẩy **bản sao** ma trận cũ vào stack, `undo` khôi phục. (3) Sinh chuỗi lệnh ngẫu nhiên có cả `undo`.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1, Bài 2
ops1 = ["SubrectangleQueries", "updateSubrectangle", "getValue", "updateSubrectangle", "getValue", "getValue"]
args1 = [[[[1, 2], [3, 4]]], [0, 1, 1, 1, 7], [1, 1], [0, 0, 0, 1, 0], [0, 1], [1, 0]]
res1 = run_ops(SubrectHistory, ops1, args1)
print("Bài 1:", res1)
assert res1 == [None, None, 7, None, 0, 3]

q = SubrectBrute([[1, 2], [3, 4]])
q.updateSubrectangle(0, 1, 1, 1, 7); s1 = [r[:] for r in q.rect]
q.updateSubrectangle(0, 0, 0, 1, 0); s2 = [r[:] for r in q.rect]
print("Bài 2:", s1, s2)
assert (s1, s2) == ([[1, 7], [3, 7]], [[0, 0], [3, 7]])

Bài 1: [None, None, 7, None, 0, 3]
Bài 2: [[1, 7], [3, 7]] [[0, 0], [3, 7]]


In [30]:
class HistoryWithRow(SubrectHistory):
    def getRow(self, row):
        out = self.rect[row][:]
        for r1, c1, r2, c2, v in self.hist:              # cũ -> mới
            if r1 <= row <= r2:
                out[c1:c2 + 1] = [v] * (c2 - c1 + 1)
        return out

class MySliceQueries:
    def __init__(self, rectangle):
        self.g = [row[:] for row in rectangle]
    def updateSubrectangle(self, r1, c1, r2, c2, v):
        vals = [v] * (c2 - c1 + 1)
        for r in range(r1, r2 + 1):
            self.g[r][c1:c2 + 1] = vals
    def getValue(self, r, c):
        return self.g[r][c]

rng = random.Random(31)
for _ in range(300):
    rows, cols = rng.randint(1, 6), rng.randint(1, 6)
    ops, args = random_ops(rows, cols, rng.randint(1, 40), rng, max_val=9)
    assert run_ops(MySliceQueries, ops, args) == run_ops(SubrectBrute, ops, args)
    h, b = HistoryWithRow(args[0][0]), SubrectBrute(args[0][0])
    for name, a in zip(ops[1:], args[1:]):
        getattr(h, name)(*a); getattr(b, name)(*a)
    for r in range(rows):
        assert h.getRow(r) == b.rect[r]
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [31]:
class SubrectUndo(SubrectHistory):
    def undo(self) -> None:
        if self.hist:
            self.hist.pop()

class UndoRef:
    def __init__(self, rectangle):
        self.g = [row[:] for row in rectangle]
        self.snap = []
    def updateSubrectangle(self, r1, c1, r2, c2, v):
        self.snap.append([row[:] for row in self.g])
        for r in range(r1, r2 + 1):
            for c in range(c1, c2 + 1):
                self.g[r][c] = v
    def getValue(self, r, c):
        return self.g[r][c]
    def undo(self):
        if self.snap:
            self.g = self.snap.pop()

rng = random.Random(41)
for _ in range(300):
    rows, cols = rng.randint(1, 5), rng.randint(1, 5)
    ops, args = random_ops(rows, cols, rng.randint(1, 40), rng, max_val=9)
    for i in range(1, len(ops)):
        if rng.random() < 0.25:
            ops[i], args[i] = "undo", []
    assert run_ops(SubrectUndo, ops, args) == run_ops(UndoRef, ops, args)
print("Bài 5: OK (300 chuỗi lệnh có undo)")

Bài 5: OK (300 chuỗi lệnh có undo)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "gán vùng – đọc điểm" bằng lịch sử:**

```python
class SubrectangleQueries:
    def __init__(self, rectangle):
        self.rect = [row[:] for row in rectangle]
        self.hist = []
    def updateSubrectangle(self, row1, col1, row2, col2, newValue):
        self.hist.append((row1, col1, row2, col2, newValue))
    def getValue(self, row, col):
        for r1, c1, r2, c2, v in reversed(self.hist):
            if r1 <= row <= r2 and c1 <= col <= c2:
                return v
        return self.rect[row][col]
```

**Checklist:**
- [ ] Sao chép **sâu** ma trận đầu vào.
- [ ] Biên **bao gồm** `row2`, `col2` (`range(..., +1)` / `<=`).
- [ ] Lịch sử duyệt **ngược** (mới nhất thắng).
- [ ] Điều kiện phủ: hàng so với `r1..r2`, cột so với `c1..c2`.
- [ ] `updateSubrectangle` trả về `None`.
- [ ] Đã test ma trận không vuông, 1×1, update chồng nhau.